In [84]:
import torch
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader
import math
import tqdm

In [5]:
class CustomDataset(Dataset):
  def __init__(self, data, transform = None,target_transform = None):
    self.data = data.data
    self.labels =data.targets
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self,idx):
    return self.data[idx],self.labels[idx]


In [144]:
class CustomDataLoader:
  def __init__(self,dataset:CustomDataset,batch_size:int ,shuffle:bool):
    self.dataset = dataset
    self.batch_size = batch_size
    self.shuffle = shuffle

  def __iter__(self):
    indices = torch.arange(len(self.dataset))

    if self.shuffle:
      indices = indices[torch.randperm(len(self.dataset))]

    for i in range(0,len(indices),self.batch_size):
      batch_indices = indices[i:i+self.batch_size]
      batch = [self.dataset[idx.item()] for idx in batch_indices]
      batch = self.collate_fn(batch)
      yield batch

  def __len__(self):
    return math.ceil(len(self.dataset)/self.batch_size)

  def collate_fn(self,samples):
    xs,ys = zip(*samples)
    xs = [torch.tensor(x/255.0,dtype=torch.float32) for x in xs]
    return torch.stack(xs), torch.tensor(ys)


In [110]:
train = MNIST(root ='.',train=True,download=True)
train

Dataset MNIST
    Number of datapoints: 60000
    Root location: .
    Split: Train

In [111]:
device = torch.accelerator.current_accelerator()
device

device(type='cuda')

In [146]:
train_dataset = CustomDataset(train)
train_dataloader = CustomDataLoader(train_dataset,batch_size=32,shuffle=True)


In [149]:
from typing import Callable
from __future__ import annotations
class AdamW(torch.optim.Optimizer):

  def __init__(self, params,lr = 1e-3,betas = (0.9,0.999),eps = 1e-8,weight_decay = 0.01):
    defaults = dict(lr = lr,betas = betas, eps =eps, weight_decay= weight_decay)
    super().__init__(params,defaults)
  def step(self, closure : Callable | None = None):

    loss = None

    if closure is not None:
      loss = closure()

    for group in self.param_groups:
      for p in group['params']:
        if p.grad.data is None:
          continue
        state= self.state[p]
        grad = p.grad.data
        if len(state) ==0 :
          state['t'] = 1
          state['m'] = torch.zeros_like(grad)
          state['v'] = torch.zeros_like(grad)

        alpha= group['lr']
        beta1,beta2 = group['betas']
        eps = group['eps']
        weight_decay = group['weight_decay']

        t = state.get('t')
        prev_mt = state.get('m')
        prev_vt = state.get('v')

        alpha_t = alpha*(math.sqrt(1-beta2**t)/(1-beta1**t))
        p.data -= alpha * weight_decay * p.data

        m_t = prev_mt* beta1 + (1-beta1)*grad
        v_t = prev_vt * beta2 + (1-beta2)*(grad**2)

        p.data -= alpha_t * (m_t/(torch.sqrt(v_t)+eps))


        state['t'] = t + 1
        state['m'] = m_t
        state['v'] = v_t

    return loss


In [148]:
class MLP(torch.nn.Module):

  def __init__(self,input_dim,hidden_dim,output_dim):
    super().__init__()
    std = 1/math.sqrt(input_dim+hidden_dim)
    self.weight1 = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.empty(input_dim,hidden_dim),std= std, a = -2*std,b=2*std),requires_grad = True)
    self.bias1 = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.empty(hidden_dim),std= std, a= -1*std,b= std),requires_grad = True)
    std = 1/math.sqrt(hidden_dim+output_dim)
    self.weight2 = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.empty(hidden_dim,output_dim),std= std, a = -2*std,b=2*std),requires_grad = True)
    self.bias2 = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.empty(output_dim),std= std, a= -1*std,b= std),requires_grad = True)

  def forward(self,x):
    x = x.view(x.size(0),-1)

    self.h = x@self.weight1 + self.bias1
    self.h_ = sigmoid(self.h)
    self.z2 = self.h_@self.weight2 + self.bias2
    # print(self.z2.shape)

    # print(self.o.shape)
    return self.z2




In [172]:
model = MLP(28*28,256,10)
model.to(device)
optimizer = AdamW(model.parameters(),lr = 1e-3)

num_epochs = 10
# 1875 batches per epoch * 10 epochs = 18750 total steps
warmup_iters = 1000
cosine_iters = 18750

min_learning_rate = 1e-6
max_learning_rate = 1e-2

def Train(model,dataloader,optimizer,num_epochs):
  global_step = 0
  for epoch in range(num_epochs):
    total_loss = 0
    for batch in tqdm.tqdm(dataloader):
      x,y = batch
      x = x.to(device)
      y = y.to(device)
      # print(y.shape)
      outputs = model(x)


      lr = cosine_lr_scheduler(global_step,warmup_iters,cosine_iters,min_learning_rate,max_learning_rate)
      for g in optimizer.param_groups:
        g['lr'] = lr

      loss = cross_entropy_loss(outputs,y)

      optimizer.zero_grad()
      dw1,db1,dw2,db2 = cross_entropy_backprop(outputs,x,y,model.weight1,model.bias1,model.weight2,model.bias2,model.h_,model.h,model.z2)

      model.weight1.grad = dw1
      model.bias1.grad = db1
      model.weight2.grad = dw2
      model.bias2.grad = db2
      clip_gradients(model.parameters(),1)
      optimizer.step()

      total_loss += loss.item()
      global_step += 1

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch : {epoch}, Avg Loss : {avg_loss:.4f}")


In [173]:
def sigmoid(x):
  # exp = torch.exp(-x)
  return torch.sigmoid(x)

def relu(x):
  return torch.max(x,torch.zeros_like(x))

def softmax(x, dim = -1):

  x_max = torch.max(x,dim = dim, keepdim =True)[0]
  rescaled = x - x_max
  exp = torch.exp(rescaled)
  sum = torch.sum(exp,dim = dim , keepdim = True)
  return exp/sum

def log_softmax(x,dim = -1):
  x_max = torch.max(x,dim = dim, keepdim = True)[0]
  rescaled = x - x_max
  sum  = torch.sum(torch.exp(rescaled),dim = dim,keepdim = True)
  return rescaled - torch.log(sum)

def cross_entropy_loss(inputs,targets):
  negative_logs = -log_softmax(inputs)
  return torch.mean(torch.gather(negative_logs, -1, targets.unsqueeze(-1)))

def cross_entropy_backprop(outputs,x,y,w1,b1,w2,b2, a1, z1,z2):
  y_one_hot =  torch.nn.functional.one_hot(y,num_classes = outputs.size(-1))
  x = x.view(x.size(0),-1)
  m = x.size(0)
  dz2 = softmax(outputs) - y_one_hot
  dw2 =(a1.T @ dz2)/m
  db2 = torch.mean(dz2,dim=0)

  da1 = dz2 @ w2.T
  dz1 = da1 * a1 * (1-a1)
  # print(dz1.shape)
  dw1 = (x.T @ dz1)/m
  db1 = torch.mean(dz1, dim = 0)
  # print(dw1.shape,dw2.shape, db1.shape, db2.shape)

  return dw1,db1,dw2,db2

def clip_gradients(parameters,max_norm):

  grads = [p.grad for p in parameters if p.grad is not None]
  norm = torch.tensor(0.0, device = grads[0].device)

  for g in grads:
    norm += torch.sum(g**2)

  norm = torch.sqrt(norm)

  clip_coeff = min(1, max_norm/(norm + 1e-6))

  for p in parameters:
    if p.grad is not None:
      p.grad.data.mul_(clip_coeff)

def cosine_lr_scheduler(it, warmup_iters, cosine_iters, min_learning_rate, max_learning_rate):

  if it < warmup_iters:
    return max_learning_rate * (it/warmup_iters)

  if it >  cosine_iters:
    return min_learning_rate

  decay = (it - warmup_iters)/(cosine_iters - warmup_iters)
  decay_rate = 0.5 * (1 + math.cos(math.pi * decay))

  return min_learning_rate + (max_learning_rate - min_learning_rate) * decay_rate




In [174]:
Train(model,train_dataloader,optimizer,num_epochs)

  0%|          | 0/1875 [00:00<?, ?it/s]/tmp/ipykernel_18291/1206519223.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs = [torch.tensor(x/255.0,dtype=torch.float32) for x in xs]
100%|██████████| 1875/1875 [00:07<00:00, 256.84it/s]


Epoch : 0, Avg Loss : 0.3654


100%|██████████| 1875/1875 [00:06<00:00, 279.44it/s]


Epoch : 1, Avg Loss : 0.1278


100%|██████████| 1875/1875 [00:06<00:00, 274.35it/s]


...

Epoch : 7, Avg Loss : 0.0090


100%|██████████| 1875/1875 [00:06<00:00, 292.78it/s]


Epoch : 8, Avg Loss : 0.0044


100%|██████████| 1875/1875 [00:07<00:00, 264.41it/s]

Epoch : 9, Avg Loss : 0.0031


In [175]:
test = MNIST(root ='.',train=False,download=True)
test

Dataset MNIST
    Number of datapoints: 10000
    Root location: .
    Split: Test

In [176]:
testdataset= CustomDataset(test)
testdataloader = CustomDataLoader(testdataset,batch_size=32,shuffle=False)

In [177]:
def evaluate(model,dataloader):
  predictions = []
  accuracy = 0
  with torch.no_grad():
    for batch in tqdm.tqdm(testdataloader):
      x, y = batch
      x = x.to(device)
      y = y.to(device)

      outputs = model(x)

      outputs = softmax(outputs)
      prediction = torch.argmax(outputs,dim = -1)

      predictions.append(prediction)
      accuracy += torch.sum(prediction == y).item()

  return predictions, accuracy/len(dataloader.dataset)

In [178]:
predictions,accuracy = evaluate(model,testdataloader)
accuracy

  0%|          | 0/313 [00:00<?, ?it/s]/tmp/ipykernel_18291/1206519223.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs = [torch.tensor(x/255.0,dtype=torch.float32) for x in xs]
100%|██████████| 313/313 [00:00<00:00, 522.22it/s]


0.9839